# Dedicated OCR Training Notebook (For Kaggle / Google Colab)
This notebook is specifically streamlined to train the Conv-Transformer Urdu OCR model. It skips the exploratory data analysis, dataset merging, and image restoration (Phase 1 & 2) to focus solely on OCR sequence training (Phase 3).

**Before you run this on Kaggle/Colab:**
1. Ensure you have uploaded the project `.py` files (`models/`, `datasets/`, `preprocessing.py`).
2. Ensure you have uploaded the splits (`splits/train.csv`, `splits/val.csv`, etc.).
3. Set `DATASET_ROOT` below to point to where the Kaggle dataset is mounted (e.g., `/kaggle/input/urdu-ocr-dataset/`).

In [ ]:
import os, sys
import pandas as pd
import torch
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
from models.vocab import Vocabulary
from models.ocr_model import build_ocr_model
from models.ocr_trainer import OCRTrainer
from datasets.ocr_dataset import get_ocr_dataloaders

# ==========================================
# HYPERPARAMETERS & CONFIGURATION
# ==========================================
DATASET_ROOT = os.getenv("DATASET_ROOT", "V:\\") # Change this for Kaggle/Colab!
BATCH_SIZE = 16
EPOCHS = 100
LR = 3e-4
BETAS = (0.9, 0.98)
EPS = 1e-9


# PATH MAPPING for Kaggle/Colab nested structures
# This replaces the local folder names in the CSV with Kaggle's nested structure
PATH_MAPPING = {
    r"MMU-OCR-21\TextLines": "MMU-OCR-21/MMU-OCR-21/TextLines",
    r"UHWR\Dataset": "UHWR/DataSet/UHWR/UHWR/Dataset"
}
# Ensure checkpoints folder exists
os.makedirs('checkpoints', exist_ok=True)


### 1. Build / Load Vocabulary
We must ensure the vocabulary is exactly the same across training sessions.

In [ ]:
vocab = Vocabulary()
vocab_path = 'checkpoints/vocab.json'

if os.path.exists(vocab_path):
    print("Loading existing vocabulary...")
    vocab.load(vocab_path)
else:
    print("Building vocabulary from scratch...")
    train_df = pd.read_csv('splits/train.csv')
    train_labels = train_df['label'].dropna().astype(str).tolist()
    chars_file = os.path.join(DATASET_ROOT, 'UHWR', 'chars.txt')
    vocab.build_from_texts(train_labels, chars_file=chars_file)
    vocab.save(vocab_path)

print(f'Vocabulary size: {vocab.size}')

### 2. Dataloaders

In [ ]:
# Load dataloaders with source-weighting configuration for handling class imbalance
ocr_loaders = get_ocr_dataloaders(
    data_dir=DATASET_ROOT,
    path_mapping=PATH_MAPPING,
    batch_size=BATCH_SIZE,
    num_workers=2  # Can increase on Kaggle/Colab
)

print(f"Train batches: {len(ocr_loaders['train'])}")
print(f"Val batches:   {len(ocr_loaders['val'])}")

### 3. Model Initialization & Resumption Logic
This cell will automatically load `final_ocr_model.pth` if it exists so you can seamlessly resume training across multiple Colab/Kaggle sessions without starting from scratch.

In [ ]:
ocr_model = build_ocr_model(
    vocab_size=vocab.size,
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=1024,
    dropout=0.1
).to(DEVICE)

final_ocr_path = os.path.join('checkpoints', 'final_ocr_model.pth')

if os.path.exists(final_ocr_path):
    print('==================================================================')
    print('FOUND EXISTING CHECKPOINT! Resuming training from these weights...')
    print('==================================================================')
    ocr_model.load_state_dict(torch.load(final_ocr_path, map_location=DEVICE, weights_only=True))
else:
    print('==================================================================')
    print('No existing final checkpoint found. Starting training from scratch...')
    print('==================================================================')

### 4. Training Loop

In [ ]:
ocr_trainer = OCRTrainer(
    model=ocr_model,
    vocab=vocab,
    device=DEVICE,
    lr=LR,
    betas=BETAS,
    eps=EPS,
)

print(f'\nTraining Conv-Transformer OCR for {EPOCHS} epochs...')
print('=' * 70)

ocr_history = ocr_trainer.fit(
    train_loader=ocr_loaders['train'],
    val_loader=ocr_loaders['val'],
    epochs=EPOCHS,
    save_dir='checkpoints',
    verbose=True,
)